# MLFF 학습 (Colab T4 GPU)

맥북 CPU로 1 epoch에 66분 걸리던 학습을 GPU로 옮기기 위한 노트북입니다.

## 실행 전 반드시 할 일 2가지

**1) GPU 켜기**

상단 메뉴 → `런타임` → `런타임 유형 변경` → 하드웨어 가속기에서 **T4 GPU** 선택 → 저장

**2) 구글 드라이브에 데이터 올리기**

드라이브에 아래 구조로 폴더를 만들고 파일 2개를 업로드하세요.

```
내 드라이브/
└── mlff-project/
    └── data/
        ├── train.xyz   (맥북의 ~/mlff-project/data/processed/train.xyz)
        └── valid.xyz   (맥북의 ~/mlff-project/data/processed/valid.xyz)
```

합쳐서 약 140MB라 업로드에 몇 분 걸립니다. **한 번만 올려두면 이후엔 계속 재사용**합니다.

> 파운데이션 모델(mace-mp small)은 노트북이 자동으로 내려받으므로 업로드하지 않아도 됩니다.

준비가 끝나면 아래 셀들을 **위에서부터 순서대로** 실행하세요 (각 셀 왼쪽의 ▶ 버튼 또는 `Shift+Enter`).

## 1. GPU가 제대로 잡혔는지 확인

In [ ]:
!nvidia-smi

import torch
print()
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
else:
    print("!! GPU가 안 잡혔습니다. 런타임 유형 변경에서 T4 GPU를 선택했는지 확인하세요.")

위 출력에 `Tesla T4`가 보이고 `CUDA 사용 가능: True`면 정상입니다.

## 2. 구글 드라이브 연결

실행하면 권한 요청 창이 뜹니다. 본인 구글 계정으로 허용해 주세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 패키지 설치

Colab은 세션이 끊기면 설치한 것이 사라지므로, 접속할 때마다 이 셀을 다시 실행해야 합니다. 2~3분 걸립니다.

In [ ]:
!pip install -q mace-torch ase

# cuequivariance: e3nn 텐서곱을 GPU에서 가속하는 NVIDIA 라이브러리.
# 맥에서는 쓸 수 없었지만 CUDA에서는 사용 가능하며 MACE 속도에 도움이 됩니다.
# 설치가 실패해도 학습 자체는 정상 진행되므로 오류가 나도 무시하고 계속 진행하세요.
!pip install -q cuequivariance-torch cuequivariance-ops-torch-cu12 || echo "cuequivariance 설치 실패 - 무시하고 진행 가능"

import mace, ase
print("설치 완료 / mace:", mace.__version__, "/ ase:", ase.__version__)

## 4. 데이터 확인

In [ ]:
import os

DRIVE_DIR = '/content/drive/MyDrive/mlff-project'
TRAIN_FILE = f'{DRIVE_DIR}/data/train.xyz'
VALID_FILE = f'{DRIVE_DIR}/data/valid.xyz'

for path in [TRAIN_FILE, VALID_FILE]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"OK  {path}  ({size_mb:.1f} MB)")
    else:
        print(f"없음 {path}  <-- 드라이브에 업로드했는지 확인하세요")

# 구조 개수도 확인 (train 17992 / valid 4518 이 나와야 정상)
from ase.io import read
print()
print("train 구조 수:", len(read(TRAIN_FILE, ':')))
print("valid 구조 수:", len(read(VALID_FILE, ':')))

## 5. 학습 설정

### v1 실행에서 배운 것 (2026-08-29)

`LOSS='huber', LR=0.01`로 50 epoch을 시도했다가 **실패**했습니다.

| epoch | 에너지 | 힘 |
|---|---|---|
| 14 | 19.5 | **337** ← 최저 |
| 28 | 20.7 | **1188** ← 3.5배 발산 |

원인이 두 가지였습니다.

**① `huber_delta=0.01`이 너무 작았습니다.** 오차가 0.01을 넘으면 벌점이 더 커지지 않아,
모델이 어려운 구조를 포기해버렸습니다. huber loss는 계속 내려가는데 실제 RMSE는 폭발하는
괴리가 생겼습니다.

**② 안전장치가 전부 꺼져 있었습니다.** MACE에는 자동 조절 기능이 있는데 기본값이 이랬습니다.

| 기능 | 기본값 | 결과 |
|---|---|---|
| `scheduler_patience` | 50 | 50 epoch 정체돼야 lr 감소 → 33 epoch 실행에선 미작동 |
| `patience` (조기 종료) | 2048 | 사실상 꺼짐 → 발산해도 안 멈춤 |
| `keep_checkpoints` | False | epoch 14 최고 모델이 덮어쓰기로 삭제됨 |

### v2 설정

| 설정 | v1 | v2 | 이유 |
|---|---|---|---|
| `LOSS` | huber | **weighted** | A/B 실험 우세 (344 vs 579). 또한 valid loss가 RMSE와 일치해야 조기 종료가 올바로 작동 |
| `LR` | 0.01 | **0.005** | 절반으로 낮춤 |
| `SCHEDULER_PATIENCE` | 50 | **3** | 3회 정체 시 학습률 20% 감소 |
| `PATIENCE` | 2048 | **10** | 10회 정체 시 중단하고 최고 모델 저장 |
| `EPOCHS` | 50 | **40 (상한)** | 실제 종료는 patience가 결정 |
| 체크포인트 | 마지막만 | **전부 보관** | 좋은 모델을 잃지 않기 위해 |

### 자동 종료 방식

epoch 수를 미리 정하지 않고, 아래 순서로 스스로 판단합니다.

```
성능 정체 3회  -> 학습률 20% 감소 (회복 시도)
성능 정체 6회  -> 학습률 추가 감소
성능 정체 10회 -> 중단, 최고 성적 모델을 최종 저장
```

포기하기 전에 학습률을 낮춰 회복할 기회를 2~3번 줍니다.

> **맥북과 다른 점**: 애플 GPU에서 막혔던 float64 문제가 없으므로 `patch_mace_mps.py`를 돌릴 필요가 없습니다.

In [ ]:
NAME = 'mace_colab_v2'

# --- 핵심 변경 ---
LR = 0.005                # v1의 0.01은 발산 -> 절반으로
LOSS = 'weighted'         # v1의 huber(delta=0.01)는 실패
SCHEDULER_PATIENCE = 3    # 3회 정체 -> 학습률 20% 감소 (회복 시도)
PATIENCE = 10             # 10회 정체 -> 중단하고 최고 모델 저장
EPOCHS = 40               # 상한선. 실제 종료 시점은 PATIENCE가 결정

FORCES_WEIGHT = 1         # MACE 기본값은 100이지만 논문(SAIT)은 1을 사용.
                          # 100이면 loss의 99%가 힘에서 나와 극단 구조에 휘둘림.
                          # 주의: 이 값을 낮춰도 RMSE_F가 100배 줄어드는 것이 아님.
                          #       로그의 loss= 숫자만 약 100배 작아짐.
ENERGY_WEIGHT = 1

# --- 유지 ---
BATCH_SIZE = 16
DTYPE = 'float32'
HUBER_DELTA = 0.01        # LOSS='huber'일 때만 사용됨

# 결과는 드라이브에 저장 -> 세션이 끊겨도 남습니다
OUT_DIR = f'{DRIVE_DIR}/runs/{NAME}'
os.makedirs(OUT_DIR, exist_ok=True)

print('결과 저장 위치:', OUT_DIR)
print(f'최대 소요 시간: 약 {EPOCHS * 10 / 60:.1f}시간 (1 epoch = 약 10분)')
print(f'-> 다만 {PATIENCE}회 연속 정체되면 그 전에 자동 종료됩니다')
print()
print('[비교 기준] v1 최고: 에너지 19.5 / 힘 337.1 (epoch 14)')
print('[목표]      논문 MACE: EF metric 155~190')

## 6. 학습 실행

**주의사항**
- 브라우저 탭을 닫으면 학습이 중단됩니다. 창은 열어두세요.
- 약 90분간 아무 조작이 없으면 연결이 끊깁니다. 가끔 화면을 확인해 주세요.
- 중간에 끊겨도 체크포인트가 드라이브에 저장되므로, 아래 **7번 셀**로 이어서 재개할 수 있습니다.

### 로그에서 볼 것

`RMSE_E_per_atom`(에너지)과 `RMSE_F`(힘) 두 숫자만 보시면 됩니다.

| 지표 | v1 최고 기록 | 목표 |
|---|---|---|
| RMSE_E_per_atom | 19.5 meV | 20 이하 유지 |
| RMSE_F | 337 meV/A | 300 아래로 |

> **`loss=` 숫자는 무시하세요.** v2는 `FORCES_WEIGHT`를 100에서 1로 바꿨기 때문에
> loss 값이 v1보다 약 100배 작게 표시됩니다. 이건 성능이 좋아진 것이 아니라
> 계산식이 바뀐 것뿐입니다. **성능 비교는 반드시 RMSE로 하세요.**

### 자동 종료

아래 문구가 뜨면 정상 종료입니다 (v1처럼 발산을 방치하지 않습니다).

```
Stopping optimization after 10 epochs without improvement
```

In [ ]:
loss_args = f'--loss={LOSS}'
if LOSS == 'huber':
    loss_args += f' --huber_delta={HUBER_DELTA}'

!mace_run_train \
  --name="{NAME}" \
  --train_file="{TRAIN_FILE}" \
  --valid_file="{VALID_FILE}" \
  --foundation_model="small" \
  --multiheads_finetuning=False \
  --energy_key="energy" \
  --forces_key="forces" \
  --E0s="average" \
  --batch_size={BATCH_SIZE} \
  --max_num_epochs={EPOCHS} \
  --lr={LR} \
  --patience={PATIENCE} \
  --scheduler_patience={SCHEDULER_PATIENCE} \
  {loss_args} \
  --energy_weight={ENERGY_WEIGHT} --forces_weight={FORCES_WEIGHT} \
  --ema --ema_decay=0.99 \
  --default_dtype={DTYPE} \
  --device=cuda \
  --seed=42 \
  --save_cpu \
  --keep_checkpoints \
  --save_all_checkpoints \
  --model_dir="{OUT_DIR}" \
  --checkpoints_dir="{OUT_DIR}/checkpoints" \
  --results_dir="{OUT_DIR}" \
  --log_dir="{OUT_DIR}"

## 7. (연결이 끊겼을 때) 이어서 학습하기

위 학습이 도중에 끊겼다면, 1~5번 셀을 다시 실행한 뒤 **이 셀**을 실행하세요.
`--restart_latest`가 드라이브에 저장된 마지막 체크포인트부터 이어서 학습합니다.

In [ ]:
loss_args = f'--loss={LOSS}'
if LOSS == 'huber':
    loss_args += f' --huber_delta={HUBER_DELTA}'

!mace_run_train \
  --name="{NAME}" \
  --train_file="{TRAIN_FILE}" \
  --valid_file="{VALID_FILE}" \
  --foundation_model="small" \
  --multiheads_finetuning=False \
  --energy_key="energy" \
  --forces_key="forces" \
  --E0s="average" \
  --batch_size={BATCH_SIZE} \
  --max_num_epochs={EPOCHS} \
  --lr={LR} \
  --patience={PATIENCE} \
  --scheduler_patience={SCHEDULER_PATIENCE} \
  {loss_args} \
  --energy_weight={ENERGY_WEIGHT} --forces_weight={FORCES_WEIGHT} \
  --ema --ema_decay=0.99 \
  --default_dtype={DTYPE} \
  --device=cuda \
  --seed=42 \
  --save_cpu \
  --keep_checkpoints \
  --save_all_checkpoints \
  --restart_latest \
  --model_dir="{OUT_DIR}" \
  --checkpoints_dir="{OUT_DIR}/checkpoints" \
  --results_dir="{OUT_DIR}" \
  --log_dir="{OUT_DIR}"

## 8. 학습 결과 정리

epoch별 오차가 어떻게 변했는지 그래프로 확인합니다.

In [ ]:
import glob, re
import matplotlib.pyplot as plt

log_files = sorted(glob.glob(f'{OUT_DIR}/*.log'))
print('로그 파일:', log_files)

epochs, e_rmse, f_rmse = [], [], []
pattern = re.compile(r'Epoch (\d+):.*RMSE_E_per_atom=\s*([\d.]+) meV, RMSE_F=\s*([\d.]+)')

for lf in log_files:
    for line in open(lf):
        m = pattern.search(line)
        if m:
            epochs.append(int(m.group(1)))
            e_rmse.append(float(m.group(2)))
            f_rmse.append(float(m.group(3)))

if epochs:
    best = min(range(len(epochs)), key=lambda i: e_rmse[i] + f_rmse[i])
    print(f"\n최고 성적: epoch {epochs[best]}")
    print(f"  에너지 RMSE: {e_rmse[best]:.2f} meV/atom")
    print(f"  힘 RMSE:     {f_rmse[best]:.2f} meV/A")
    print(f"  EF metric:   {e_rmse[best] + f_rmse[best]:.2f}  (논문 MACE 기준 155~190)")

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epochs, e_rmse, marker='o', ms=3)
    ax[0].set_title('Energy RMSE (meV/atom)'); ax[0].set_xlabel('epoch'); ax[0].grid(alpha=.3)
    ax[1].plot(epochs, f_rmse, marker='o', ms=3, color='tab:orange')
    ax[1].set_title('Force RMSE (meV/A)'); ax[1].set_xlabel('epoch'); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print('로그에서 epoch 기록을 찾지 못했습니다. 학습이 진행됐는지 확인하세요.')

## 9. 학습된 모델 내려받기

결과 모델은 이미 드라이브(`runs/{NAME}/`)에 저장되어 있습니다.
맥북으로 가져와서 예측(`07_predict_test.py`)에 쓰려면 아래에서 파일 위치를 확인한 뒤,
구글 드라이브 웹/앱에서 내려받아 맥북의 `~/mlff-project/models/`에 넣으면 됩니다.

In [ ]:
!ls -la "{OUT_DIR}"
print()
print('이 중 .model 파일을 드라이브에서 내려받아 맥북의 ~/mlff-project/models/ 에 넣으세요.')